In [ ]:
import os
import sys

# 1. Ensure driver classpath inherits the Iceberg JAR
ICEBERG_JAR_PATH = "C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar"
os.environ["PYSPARK_SUBMIT_ARGS"] = f'--conf spark.driver.extraClassPath="{ICEBERG_JAR_PATH}" pyspark-shell'
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# 2. R2 Configuration
R2_AID=""
R2_AK=""
R2_SAK=""
R2_BUCKET_NAME = "wide-world-importers-dw"

# 3. Initialize PySpark connected to Cloudflare R2
spark = (
    SparkSession.builder
    .appName("Jupyter Cloudflare R2 Inspector")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.r2_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.r2_catalog.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
    .config("spark.sql.catalog.r2_catalog.warehouse", f"s3a://{R2_BUCKET_NAME}/iceberg")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{R2_AID}.r2.cloudflarestorage.com")
    .config("spark.hadoop.fs.s3a.access.key", R2_AK)
    .config("spark.hadoop.fs.s3a.secret.key", R2_SAK)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)

print("✅ Connected to Cloudflare R2!\n")

In [ ]:
spark.sql("""
UPDATE r2_catalog.dimension.date
SET 
    Date = date_add(Date, 57),
    Day_Number = day(date_add(Date, 57)),
    Day = date_format(date_add(Date, 57), 'EEEE'),
    Month = date_format(date_add(Date, 57), 'MMMM'),
    Short_Month = date_format(date_add(Date, 57), 'MMM'),
    Calendar_Month_Number = month(date_add(Date, 57)),
    Calendar_Month_Label = date_format(date_add(Date, 57), 'yyyy-MMM'),
    Calendar_Year = year(date_add(Date, 57)),
    Calendar_Year_Label = concat('CY', year(date_add(Date, 57))),
    ISO_Week_Number = weekofyear(date_add(Date, 57))
""")

spark.sql("""
UPDATE r2_catalog.fact.transaction
SET 
    Date_Key = date_add(Date_Key, 57)
""")

spark.sql("""
UPDATE r2_catalog.fact.movement
SET 
    Date_Key = date_add(Date_Key, 57)
""")

spark.sql("""
UPDATE r2_catalog.fact.order
SET 
    Order_Date_Key = date_add(Order_Date_Key, 57),
    Picked_Date_Key = date_add(Picked_Date_Key, 57)
""")

spark.sql("""
UPDATE r2_catalog.fact.purchase
SET 
    Date_Key = date_add(Date_Key, 57)
""")

spark.sql("""
UPDATE r2_catalog.fact.sale
SET 
    Invoice_Date_Key = date_add(Invoice_Date_Key, 57),
    Delivery_Date_Key = date_add(Delivery_Date_Key, 57)
""")

# Max Date  = 07/27/2026

In [ ]:
# 1. List all namespaces on R2
spark.sql("SHOW NAMESPACES IN r2_catalog").show()

# 2. Check tables in the 'fact' namespace (try both lowercase and camel case)
try:
    spark.sql("SHOW TABLES IN r2_catalog.fact").show()
except Exception:
    spark.sql("SHOW TABLES IN r2_catalog.Fact").show()

In [ ]:
df_purchase = spark.table("r2_catalog.fact.purchase").alias("purchase")
df_stock_holding = spark.table("r2_catalog.fact.stock_holding").alias("stock_holding")
df_order = spark.table("r2_catalog.fact.order").alias("order")
df_movement = spark.table("r2_catalog.fact.movement").alias("movement")
df_sale = spark.table("r2_catalog.fact.sale").alias("sale")
df_payment_method = spark.table("r2_catalog.dimension.payment_method").alias("payment_method")
df_supplier = spark.table("r2_catalog.dimension.supplier").alias("supplier")
df_city = spark.table("r2_catalog.dimension.city").alias("city")
df_stock_item = spark.table("r2_catalog.dimension.stock_item").alias("stock_item")
df_customer = spark.table("r2_catalog.dimension.customer").alias("customer")
df_date = spark.table("r2_catalog.dimension.date").alias("date")
df_transaction_type = spark.table("r2_catalog.dimension.transaction_type").alias("transaction_type")
df_employee = spark.table("r2_catalog.dimension.employee").alias("employee")
df_transaction = spark.table("r2_catalog.fact.transaction").alias("transaction")

In [ ]:
df_date.show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import max, min

df_date.agg(
    min("Date").alias("min_date_id"), max("Date").alias("max_date_id")
).show()

df_sale.agg(
    min("Invoice_Date_Key").alias("min_Invoice_Date_Key"), max("Invoice_Date_Key").alias("max_Invoice_Date_Key")
).show()

In [ ]:
spark.sql("""
UPDATE r2_catalog.dimension.date
SET 
    Date = add_months(Date, 120),
    Day_Number = day(add_months(Date, 120)),
    Day = date_format(add_months(Date, 120), 'EEEE'),
    Month = date_format(add_months(Date, 120), 'MMMM'),
    Short_Month = date_format(add_months(Date, 120), 'MMM'),
    Calendar_Month_Number = month(add_months(Date, 120)),
    Calendar_Month_Label = date_format(add_months(Date, 120), 'yyyy-MMM'),
    Calendar_Year = year(add_months(Date, 120)),
    Calendar_Year_Label = concat('CY', year(add_months(Date, 120))),
    ISO_Week_Number = weekofyear(add_months(Date, 120))
""")

In [ ]:

df_transaction.printSchema()
spark.sql("""
UPDATE r2_catalog.fact.transaction
SET 
    Date_Key = add_months(Date_Key, 120)
""")

In [ ]:
df_movement.printSchema()

spark.sql("""
UPDATE r2_catalog.fact.movement
SET 
    Date_Key = add_months(Date_Key, 120)
""")

In [ ]:
df_order.printSchema()

spark.sql("""
UPDATE r2_catalog.fact.order
SET 
    Order_Date_Key = add_months(Order_Date_Key, 120),
    Picked_Date_Key = add_months(Picked_Date_Key, 120)
""")


In [ ]:
df_purchase.printSchema()

spark.sql("""
UPDATE r2_catalog.fact.purchase
SET 
    Date_Key = add_months(Date_Key, 120)
""")

In [ ]:
df_sale.printSchema()

spark.sql("""
UPDATE r2_catalog.fact.sale
SET 
    Invoice_Date_Key = add_months(Invoice_Date_Key, 120),
    Delivery_Date_Key = add_months(Delivery_Date_Key, 120)
""")

In [7]:
spark.stop()